# Notebook 3b – MER-TRANS 2026 Inference: AFS (6 blocks)

**Use this notebook for:** AFS
**Supported datasets:** Spanish test set, Italian test set, FEINA sample

**Block strategy:** 346 segments split into 6 blocks of ~58 segments each due to the
longer AFS prompt. Each block uses a separate API key.

---

### Configuration guide — what to change per dataset

| Dataset | `DATASET_PATH` | `OUTPUT_DIR` | `ACTIVE_PROMPT` | `MARKER` |
|---|---|---|---|---|
| **ES test** | `data/test/es_test_data.csv` | `predictions_test_es/` | `PROMPT_AFS` | `'Texto simplificado:'` |
| **IT test** | `data/test/it_test_data.csv` | `predictions_test_it/` | `PROMPT_AFS_IT` | `'Testo semplificato:'` |
| **FEINA** | `data/feina/feina_sample_346.csv` | `predictions_feina/` | `PROMPT_AFS` | `'Texto simplificado:'` |

## Installation

In [ ]:
!pip install groq pandas tqdm

## 1. Global configuration

> Set `DATASET_PATH`, `OUTPUT_DIR`, `ACTIVE_PROMPT`, and `MARKER`
> according to the table above before running any block.

In [ ]:
import os
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

MODEL      = "llama-3.3-70b-versatile"
TEMP       = 0.3
MAX_TOKENS = 300
DELAY      = 1.0

# Change these variables according to the configuration guide above
DATASET_PATH  = 'data/test/es_test_data.csv'
OUTPUT_DIR    = 'predictions_test_es/'
ACTIVE_PROMPT = None  
MARKER        = 'Texto simplificado:'   # 'Texto simplificado:' (ES/FEINA) | 'Testo semplificato:' (IT)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Model:   {MODEL}')
print(f'Dataset: {DATASET_PATH}')
print(f'Output:  {OUTPUT_DIR}')

## 2. Prompt definitions — Advanced Few-Shot (AFS)

In [ ]:
# Spanish AFS prompt

PROMPT_AFS = """Eres un experto en simplificación de textos en español para personas con dificultades de comprensión lectora, como personas con discapacidad intelectual, bajo nivel de alfabetización o conocimiento limitado del idioma. Tu objetivo es transformar textos complejos en versiones claras, sencillas y accesibles, preservando siempre el significado original completo.

REGLA ABSOLUTA: Tu única tarea es reescribir el texto de entrada con palabras más simples. NUNCA añadas contenido que no esté literalmente en el texto de entrada. Si el texto es una pregunta, simplifica la pregunta, no la respondas.

Sigue estas instrucciones:
- Usa frases cortas con una sola idea cada una. Separa las ideas en oraciones distintas. No fragmentes en exceso: si dos ideas están relacionadas, puedes mantenerlas en una sola frase.
- Elimina los incisos y aposiciones. Conviértelas en oraciones independientes que vayan justo después.
- Usa el presente de indicativo y la voz activa. Evita la voz pasiva, los gerundios, el subjuntivo y los tiempos compuestos. Evita también la pasiva refleja (se vende, se garantiza).
- Respeta el tiempo verbal del texto original cuando narre hechos pasados.
- Elimina los conectores complejos. Sustituye "por lo tanto" y "por consiguiente" por "por eso"; "sin embargo" y "no obstante" por "pero" o "aunque".
- Usa palabras sencillas y de uso frecuente. Sustituye los términos técnicos, abstractos o poco comunes.
- NUNCA expliques palabras cotidianas ni organismos conocidos. Solo explica siglas y tecnicismos legales poco conocidos en una oración independiente justo después.
- Evita la doble negación.
- Transforma las construcciones nominales en oraciones con verbo activo.
- Cuando enumeres tres o más elementos, usa lista con guiones (–).
- Conserva toda la información del texto original.
- Entrega únicamente el texto simplificado. No escribas notas ni ofrezcas alternativas.

RECUERDA: NUNCA añadas contenido que no esté en el texto de entrada.

A continuación tienes tres ejemplos del estilo de simplificación esperado:

--- EJEMPLO 1 ---
Original: En España los abogados han de ser licenciados en Derecho y estar inscritos en el correspondiente Colegio Profesional.
Simplificado: Los abogados deben ser licenciados en Derecho y estar inscritos en un colegio profesional. Un colegio profesional es un registro público en el que los profesionales se pueden inscribir.

--- EJEMPLO 2 ---
Original: Su intervención en la Administración Justicia, actuando ante los Juzgados y Tribunales, es esencial para el correcto desarrollo de los procedimientos al asesorar a los ciudadanos e intervinientes sobre los distintos trámites procesales, garantizando así su derecho a la defensa y asistencia Letrada que consagra el artículo 24 de la Constitución.
Simplificado: La función de los abogados y letrados es esencial para que los procesos judiciales y administrativos en juzgados y tribunales sean correctos. Ellos ayudan a los ciudadanos a realizar trámites y aseguran que los ciudadanos puedan defenderse y tener apoyo legal en un juicio. Este derecho está en el artículo 24 de la Constitución española.

--- EJEMPLO 3 ---
Original: Abogados o letrados\n\nLos abogados o letrados son aquellos profesionales encargados de la defensa y dirección de las partes en los procedimientos judiciales y administrativos, asumiendo también con carácter amplio labores de asesoramiento y consejo jurídico en el ámbito extrajudicial para evitar conflictos.
Simplificado: Debes contactar con abogados o letrados. Los abogados o letrados son los profesionales encargados de defender y dirigir los procesos judiciales y administrativos. Ellos también pueden asesorarte, orientarte y dar consejos para evitar conflictos fuera del proceso judicial.

Ahora simplifica el siguiente texto aplicando las mismas reglas y el mismo estilo de los ejemplos.

Texto a simplificar:
{texto}

Texto simplificado:
"""

# Italian AFS prompt

PROMPT_AFS_IT = """Sei un esperto nella semplificazione di testi in italiano per persone con difficoltà di comprensione della lettura, come persone con disabilità intellettiva, basso livello di alfabetizzazione o conoscenza limitata della lingua. Il tuo obiettivo è trasformare testi complessi in versioni chiare, semplici e accessibili, preservando sempre il significato originale completo.

REGOLA ASSOLUTA: Il tuo unico compito è riscrivere il testo di input con parole più semplici. NON aggiungere mai contenuti che non siano letteralmente presenti nel testo di input. Se il testo è una domanda, semplifica la domanda, non risponderla.

Segui queste istruzioni:
- Usa frasi brevi con una sola idea ciascuna. Separa le idee in frasi distinte. Non frammentare eccessivamente: se due idee sono correlate, puoi mantenerle in una sola frase.
- Elimina le proposizioni incidentali e le apposizioni. Trasformale in frasi indipendenti che seguano immediatamente.
- Usa il presente indicativo e la forma attiva. Evita la forma passiva, i gerundi, il congiuntivo e i tempi composti. Evita anche il passivo riflessivo.
- Rispetta il tempo verbale del testo originale quando racconta fatti passati.
- Elimina i connettivi complessi. Sostituisci "pertanto" e "di conseguenza" con "quindi"; "tuttavia" e "nonostante" con "ma" o "anche se".
- Usa parole semplici e di uso comune. Sostituisci i termini tecnici, astratti o poco comuni.
- NON spiegare parole quotidiane, organismi noti o concetti politici generali. Spiega solo sigle e tecnicismi poco conosciuti in una frase indipendente subito dopo.
- Evita la doppia negazione.
- Trasforma le costruzioni nominali in frasi con verbo attivo.
- Quando elenchi tre o più elementi, introducili con i due punti (:) e scrivi ogni elemento su una riga separata con un trattino (–).
- Conserva tutte le informazioni del testo originale. Non aggiungere informazioni nuove.
- Consegna solo il testo semplificato. Non scrivere note né offrire alternative.

RICORDA: NON aggiungere mai contenuti che non siano presenti nel testo di input.

Di seguito trovi tre esempi dello stile di semplificazione atteso:

--- ESEMPIO 1 ---
Originale: Per capire quanto conti ancora oggi il mondo religioso, abbiamo realizzato un'inchiesta in due articoli: questo che state leggendo è dedicato ai partiti al governo, mentre qui trovate l'altro articolo, dedicato ai partiti all'opposizione.
Semplificato: Vogliamo capire quanto la religione è importante in politica oggi. Per questo, abbiamo scritto 2 articoli.
In questo articolo parliamo dei partiti al governo. L'altro articolo è sui partiti che non sono al governo. Lo trovate qui.

--- ESEMPIO 2 ---
Originale: Negli ultimi anni, a destra, la religione è tornata ad affacciarsi sulla scena politica in modo esplicito e a tratti spettacolare.
Semplificato: Negli ultimi anni, i partiti di destra parlano di nuovo molto di religione. Lo fanno in modo diretto e spesso molto forte.

--- ESEMPIO 3 ---
Originale: Erano segnali chiari: la destra italiana voleva tornare a parlare il linguaggio della tradizione, e la religione cattolica offriva simboli potenti per farlo.
Semplificato: Era un messaggio chiaro: i partiti di destra vogliono usare parole e idee tradizionali. La religione cattolica ha simboli forti per farlo.

Ora semplifica il seguente testo applicando le stesse regole e lo stesso stile degli esempi.

Testo da semplificare:
{texto}

Testo semplificato:
"""

# Select active prompt
ACTIVE_PROMPT = PROMPT_AFS   # PROMPT_AFS (ES/FEINA) | PROMPT_AFS_IT (IT)

print(f'AFS prompt loaded ({len(ACTIVE_PROMPT)} chars).')

## 3. Load dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
df = df.dropna(subset=['original_text']).copy()
df['original_text'] = df['original_text'].astype(str).str.strip()
df = df[df['original_text'].str.len() > 0].reset_index(drop=True)

print(f'Segments loaded: {len(df)}')
df.head(3)

## 4. Simplification function

In [ ]:
def extract_simplification(response: str, marker: str) -> str:
    if marker in response:
        return response.split(marker)[-1].strip()
    return response.strip()


def simplify(text: str, groq_client, model: str = MODEL, max_retries: int = 3) -> str:
    full_prompt = ACTIVE_PROMPT.format(texto=text)
    for attempt in range(1, max_retries + 1):
        try:
            resp = groq_client.chat.completions.create(
                model=model,
                temperature=TEMP,
                max_tokens=MAX_TOKENS,
                messages=[{'role': 'user', 'content': full_prompt}]
            )
            return extract_simplification(resp.choices[0].message.content, MARKER)
        except Exception as e:
            print(f'    [Attempt {attempt}/{max_retries}] Error: {e}')
            time.sleep(DELAY * attempt * 2)
    return '[ERROR]'

print('simplify() defined.')

## 5. BLOCK 1 – Segments 0–57

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_1 = 'YOUR_GROQ_API_KEY_HERE'
client_1 = Groq(api_key=GROQ_API_KEY_1)

block_1 = df.iloc[0:58].copy()
results_1 = []
errors_1 = 0

print(f'Processing {len(block_1)} segments (0–57)...')

for i, row in tqdm(block_1.iterrows(), total=len(block_1)):
    simplified = simplify(row['original_text'], client_1)
    if simplified == '[ERROR]':
        errors_1 += 1
    results_1.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_1 = pd.DataFrame(results_1)
path_1 = f'{OUTPUT_DIR}partial_block1_AFS.csv'
df_block_1.to_csv(path_1, index=False)
print(f'Block 1 done. Errors: {errors_1}/{len(block_1)}. Saved to {path_1}')

## 6. BLOCK 2 – Segments 58–115

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_2 = 'YOUR_GROQ_API_KEY_HERE'
client_2 = Groq(api_key=GROQ_API_KEY_2)

block_2 = df.iloc[58:116].copy()
results_2 = []
errors_2 = 0

print(f'Processing {len(block_2)} segments (58–115)...')

for i, row in tqdm(block_2.iterrows(), total=len(block_2)):
    simplified = simplify(row['original_text'], client_2)
    if simplified == '[ERROR]':
        errors_2 += 1
    results_2.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_2 = pd.DataFrame(results_2)
path_2 = f'{OUTPUT_DIR}partial_block2_AFS.csv'
df_block_2.to_csv(path_2, index=False)
print(f'Block 2 done. Errors: {errors_2}/{len(block_2)}. Saved to {path_2}')

## 7. BLOCK 3 – Segments 116–173

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_3 = 'YOUR_GROQ_API_KEY_HERE'
client_3 = Groq(api_key=GROQ_API_KEY_3)

block_3 = df.iloc[116:174].copy()
results_3 = []
errors_3 = 0

print(f'Processing {len(block_3)} segments (116–173)...')

for i, row in tqdm(block_3.iterrows(), total=len(block_3)):
    simplified = simplify(row['original_text'], client_3)
    if simplified == '[ERROR]':
        errors_3 += 1
    results_3.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_3 = pd.DataFrame(results_3)
path_3 = f'{OUTPUT_DIR}partial_block3_AFS.csv'
df_block_3.to_csv(path_3, index=False)
print(f'Block 3 done. Errors: {errors_3}/{len(block_3)}. Saved to {path_3}')

## 8. BLOCK 4 – Segments 174–231

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_4 = 'YOUR_GROQ_API_KEY_HERE'
client_4 = Groq(api_key=GROQ_API_KEY_4)

block_4 = df.iloc[174:232].copy()
results_4 = []
errors_4 = 0

print(f'Processing {len(block_4)} segments (174–231)...')

for i, row in tqdm(block_4.iterrows(), total=len(block_4)):
    simplified = simplify(row['original_text'], client_4)
    if simplified == '[ERROR]':
        errors_4 += 1
    results_4.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_4 = pd.DataFrame(results_4)
path_4 = f'{OUTPUT_DIR}partial_block4_AFS.csv'
df_block_4.to_csv(path_4, index=False)
print(f'Block 4 done. Errors: {errors_4}/{len(block_4)}. Saved to {path_4}')

## 9. BLOCK 5 – Segments 232–289

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_5 = 'YOUR_GROQ_API_KEY_HERE'
client_5 = Groq(api_key=GROQ_API_KEY_5)

block_5 = df.iloc[232:290].copy()
results_5 = []
errors_5 = 0

print(f'Processing {len(block_5)} segments (232–289)...')

for i, row in tqdm(block_5.iterrows(), total=len(block_5)):
    simplified = simplify(row['original_text'], client_5)
    if simplified == '[ERROR]':
        errors_5 += 1
    results_5.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_5 = pd.DataFrame(results_5)
path_5 = f'{OUTPUT_DIR}partial_block5_AFS.csv'
df_block_5.to_csv(path_5, index=False)
print(f'Block 5 done. Errors: {errors_5}/{len(block_5)}. Saved to {path_5}')

## 10. BLOCK 6 – Segments 290–end

> **Insert your Groq API key before running this cell.**

In [ ]:
GROQ_API_KEY_6 = 'YOUR_GROQ_API_KEY_HERE'
client_6 = Groq(api_key=GROQ_API_KEY_6)

block_6 = df.iloc[290:].copy()
results_6 = []
errors_6 = 0

print(f'Processing {len(block_6)} segments (290–end)...')

for i, row in tqdm(block_6.iterrows(), total=len(block_6)):
    simplified = simplify(row['original_text'], client_6)
    if simplified == '[ERROR]':
        errors_6 += 1
    results_6.append({
        'document_id':          row['document_id'],
        'original_sentence_id': row['original_sentence_id'],
        'system':               simplified,
    })
    time.sleep(DELAY)

df_block_6 = pd.DataFrame(results_6)
path_6 = f'{OUTPUT_DIR}partial_block6_AFS.csv'
df_block_6.to_csv(path_6, index=False)
print(f'Block 6 done. Errors: {errors_6}/{len(block_6)}. Saved to {path_6}')

## 11. Final merge

> Run this cell after completing all 6 blocks.

In [ ]:
paths = [f'{OUTPUT_DIR}partial_block{b}_AFS.csv' for b in range(1, 7)]
df_final = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

assert len(df_final) == len(df), f'Expected {len(df)} rows, got {len(df_final)}'

output_path = f'{OUTPUT_DIR}predictions_AFS.csv'
df_final.to_csv(output_path, index=False)

print(f'Output file: {output_path}')
print(f'Total segments: {len(df_final)}')
print(f'Errors: {(df_final["system"] == "[ERROR]").sum()}')